In [ ]:
%%writefile ambiente.h
// Ambiente subacqueo: esplorazione dalla superficie verso il fondo. La lentezza F = 1/v e' assegnata: acqua a strati,
// uno strato lento (termoclino) e due zone di acqua densa. Contiene: parametri, oggetti, maschera, lentezza
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>
#include <algorithm>

// Forme con cui si descrive l'ambiente (coordinate nel cubo [-1, 1]^3, z = quota: -1 fondo, 1 superficie)
struct Punto { double x, y, z; };
struct Sfera { double cx, cy, cz, r; };              // ostacolo sferico: centro (cx, cy, cz) e raggio r

// AMBIENTE: per cambiarlo si modifica solo questa parte
const double MARGINE = 0.04;                           // distanza minima dagli oggetti (raggio del veicolo + sicurezza)
const Punto A = { -0.8,  0.3,  0.8 };                  // partenza del veicolo: vicino alla superficie
const Punto B = {  0.75, 0.2, -0.85 };                 // arrivo (sorgente dell'iconale): sul fondo, accanto al relitto

// Acqua: 1) velocita' di base che cresce verso la superficie, v = V0 + G z (0.6 sul fondo, 1.4 in superficie);
// 2) termoclino: uno strato orizzontale a quota Z_TERMO dove l'acqua e' piu' lenta (fino al RALL_TERMO in meno);
// 3) zone di acqua densa (torbida o piu' fredda): sfere sfumate dove l'acqua e' piu' lenta (fino a rall in meno al centro)
const double V0 = 1.0, G = 0.4;
const double Z_TERMO = 0.1, S_TERMO = 0.12, RALL_TERMO = 0.6;   // quota, semispessore, rallentamento del termoclino
struct AcquaDensa { Punto c; double r, rall; };                 // centro, raggio, rallentamento al centro
const std::vector<AcquaDensa> ACQUA_DENSA = { { { -0.30, 0.25, 0.45 }, 0.25, 0.7 },
                                              { {  0.35, 0.00, -0.30 }, 0.25, 0.7 } };

// Oggetti in acqua: sfere (rocce) tra A e B, sulla strada che il veicolo farebbe senza di loro
inline std::vector<Sfera> OSTACOLI = { {  0.00, 0.40,  0.62, 0.15 },     // in alto, nell'acqua veloce
                                       {  0.48, 0.35,  0.00, 0.15 },     // nel termoclino
                                       {  0.68, 0.27, -0.45, 0.12 } };   // sopra B, verso il fondo

// Velocita' dell'acqua nel punto (x, y, z): il prodotto dei tre effetti
inline double velocita(double x, double y, double z)
{
    double v = V0 + G * z;                                                         // strati
    const double t = (z - Z_TERMO) / S_TERMO;
    v *= 1.0 - RALL_TERMO * std::exp(-t * t);                                      // termoclino
    for (const auto& a : ACQUA_DENSA) {                                            // zone di acqua densa
        const double d2 = (x - a.c.x) * (x - a.c.x) + (y - a.c.y) * (y - a.c.y) + (z - a.c.z) * (z - a.c.z);
        v *= 1.0 - a.rall * std::exp(-d2 / (a.r * a.r));
    }
    return v;
}
inline double lentezza(double x, double y, double z) { return 1.0 / velocita(x, y, z); }

// Distanza del punto (x, y, z) dalla superficie della sfera os (negativa se il punto e' dentro)
inline double distanza(const Sfera& os, double x, double y, double z)
{
    return std::sqrt((x - os.cx) * (x - os.cx) + (y - os.cy) * (y - os.cy) + (z - os.cz) * (z - os.cz)) - os.r;
}

// Vero se il punto (x, y, z) e' dentro un oggetto o a meno di MARGINE dalla sua superficie
inline bool ostacolo(double x, double y, double z)
{
    for (const auto& os : OSTACOLI) if (distanza(os, x, y, z) < MARGINE) return true;
    return false;
}

// Vero se il punto (x, y, z) e' dentro una sfera vera (senza il margine): il veicolo non ci deve mai entrare
inline bool dentro_un_oggetto(double x, double y, double z)
{
    for (const auto& os : OSTACOLI) if (distanza(os, x, y, z) < 0.0) return true;
    return false;
}

// Maschera della griglia: libero = 1, ostacolo = 0 (resta INF). Un nodo e' ostacolo se dista dal centro di una sfera
// meno di r + MARGINE: il margine (raggio del veicolo + sicurezza) e' gia' compreso
inline std::vector<unsigned char> costruisci_maschera(int N)
{
    const double h = 2.0 / (N - 1);
    const int n = N * N * N;
    std::vector<unsigned char> mask(n);
    for (int q = 0; q < n; ++q) {                        // un nodo alla volta
        const int j = q % N, i = (q / N) % N, k = q / (N * N);
        mask[q] = ostacolo(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h) ? 0 : 1;
    }
    return mask;
}

// Lentezza F = 1/v su tutta la griglia (preparata sulla CPU, poi copiata sulla GPU)
inline std::vector<float> costruisci_lentezza(int N)
{
    const double h = 2.0 / (N - 1);
    std::vector<float> F((size_t)N * N * N);
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j)
                F[((size_t)k * N + i) * N + j] = (float)lentezza(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h);
    return F;
}

// Controllo iniziale: A e B non devono stare dentro un oggetto
inline bool ambiente_valido() { return !ostacolo(A.x, A.y, A.z) && !ostacolo(B.x, B.y, B.z); }
// Dal punto (x, y, z) all'indice del nodo piu' vicino (serve per A e B)
inline int nodo(int N, double x, double y, double z)
{
    const double h = 2.0 / (N - 1);
    const int j = (int)lround((x + 1.0) / h), i = (int)lround((y + 1.0) / h), k = (int)lround((z + 1.0) / h);
    return (k * N + i) * N + j;
}

In [ ]:
%%writefile godunov3d.h
#pragma once           // il file viene incluso una sola volta
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <cmath>       // fminf, sqrtf

const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda
// Risolutore locale 3D (Godunov, primo ordine), comune ai tre metodi.
// __host__ __device__: compilata per la CPU (la usa il FMM) e per la GPU (Block FIM e FIM-Jacobi)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include "godunov3d.h"  // risolutore locale

typedef std::pair<float, int> Entry;     // coppia (u, indice del nodo) per la coda di priorita' del FMM

// Nuovo u del nodo (k, i, j) per il FMM: minimo dei vicini per asse + risolutore locale
float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
// Fast Marching Method (Alg. 3) su CPU: il riferimento per il Block FIM
// Fissa i nodi in ordine di tempo crescente con una coda di priorita'
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, const std::vector<int>& srcs, const std::vector<unsigned char>& mask)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    for (int s : srcs){heap.push(Entry(u[s], s)); state[s] = ATTIVO;}

    const int dk[6] = {  0, 0,  0, 0, -1, 1 };       // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1,  0, 0 };       // sinistra, destra, sotto, sopra,
    const int dj[6] = { -1, 1,  0, 0,  0, 0 };       // davanti, dietro

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO || !mask[nb]) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}

In [ ]:
%%writefile block_fim3d.cuh
#pragma once           // il file viene incluso una sola volta
// Block FIM 3D (Alg. 7) sulla GPU: blocchi 8^3 in shared memory, uscita anticipata, maschera degli ostacoli
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <algorithm>       // std::min
#include <cuda_runtime.h>
#include "godunov3d.h"  // risolutore locale

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo
// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

// Aggiorna i blocchi di una lista (Alg. 7): lista dei blocchi attivi (Step 1, niter iterazioni) 
// oppure dei candidati (Step 2, 1 iterazione)
__global__ void update_blocks(const int* list, const int* quantity, int niter, float* u, const float* F,
                             int N, int nbx, float h, const unsigned char* mask, float eps, unsigned char* Cb_out, int* is_candidate)
{
    // quantity = 0/1; *quantity = numero di blocchi attivi
    if (quantity && blockIdx.x >= *quantity) return;
    // (es. N = 33) giro: 1 -> blocchi CUDA lanciati: 1 -> blockIdx.x: 0 -> block = active_list[blockIdx.x]: 62
    // (es. N = 33) giro: 3 -> blocchi CUDA lanciati: 6 -> blockIdx.x: 0, 1, ..., 5 -> block = active_list[blockIdx.x]: 61, 63, 57, 67, 37, 87
    const int block = list[blockIdx.x];

    // Allocazione statica di shared memory
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];              
    // posizione del thread nel suo blocco CUDA: ognuno da 0 a 7
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    // bj: asse x , bi: asse y e bk: asse z; identifica blocco nella griglia
    int bj = block % nbx, bi = (block / nbx) % nbx, bk = block / (nbx * nbx); 
    // numero del thread dentro il blocco; 
    int tid = (tz*BLOCK + ty) * BLOCK + tx; // 0, ..., 511
    // posizione globale rispetto x,y e z: lungo tutta la griglia
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;
    int idx = (k * N + i) * N + j;  // 0, ..., N^3-1
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || !mask[idx];             // nodo da non aggiornare: fuori dalla griglia o maschera
    float f = inside ? F[idx] : 1.0f;
    // Global -> Shared
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        // posizione nella shared memory
        int sz = m / ((BLOCK + 2) * (BLOCK + 2)), sy = (m / (BLOCK + 2)) % (BLOCK + 2), sx = m % (BLOCK + 2);
        // coordinate globali della cella (sz, sy, sx): -1 perché c'è l'halo                          
        int gi = bi * BLOCK + sy - 1, gj = bj * BLOCK + sx - 1, gk = bk * BLOCK + sz - 1;
        
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();
    bool conv = true;
    for (int it = 0; it < niter; ++it){
        // p: valore del nodo all'inizio di questa iterazione
        float p = s_u[tz+1][ty+1][tx+1];
        // q: nuovo valore calcolato in questa iterazione
        float q = fixed ? p : solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);  
        __syncthreads();
        s_u[tz+1][ty + 1][tx + 1] = q;
        conv = fabsf(p - q) < eps;
        if (!__syncthreads_or(!conv)) break;   // tutti hanno scritto; se nessun nodo e' cambiato, esci
    }
    // Shared -> Global: ogni thread riscrive il proprio nodo
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];
    // Cb del blocco: vero solo se TUTTI i 512 nodi sono convergenti. __syncthreads_and fa l'AND di conv su tutti i thread del
    // blocco in una sola istruzione (al posto della riduzione ad albero della tesi, Fig. 3.7)
    const bool blocco_conv = __syncthreads_and(conv);
    if (tid == 0) Cb_out[block] = blocco_conv;
    // solo candidati: pronto per il giro dopo
    if (is_candidate && tid == 0) is_candidate[block] = 0;  
}

// costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* in_list,
                                  int* active_list, int* n_active, int n_blocks) 
{
    // blockIdx: blocco nel lancio in corso; blockDim (monodimensionale CUDA): 256; threadIdx:0,...,255
    int b = blockIdx.x * blockDim.x + threadIdx.x;           // numero del thread(blocco) su tutto il lancio
    // N = 65 -> nbx = 9 -> n_blocks = 729 -> list_blocks = 3 -> thread lanciati = 768 -> thread in eccesso = 39 
    if (b >= n_blocks) return;                               // thread(blocco) in eccesso lanciati da CUDA                                   
    in_list[b] = !Cb[b];                                      // fotografia della lista A di questo giro
    if (!Cb[b]) active_list[atomicAdd(n_active, 1)] = b;               // se NON convergente aggiungi thread(blocco) alla lista A
}
// Step 2: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* active_list, int n_active, int nbx, const unsigned char* in_list,
                                     const unsigned char* Cb_step1, unsigned char* Cb,
                                     int* is_candidate, int* cand_list, int* cand_count)
{
    int pos = blockIdx.x * blockDim.x + threadIdx.x;
    // se è maggiore o uguale al numero di blocchi attivi fai return
    if (pos >= n_active) return;                       
    int b = active_list[pos];         // un blocco della lista A; N = 33 -> A ={61, 63, ...,87}; active_list[0] =61
    Cb[b] = Cb_step1[b];             // leggi il valore di Cb
    if (!Cb_step1[b]) return;        // proseguono solo i blocchi appena convergenti
    // bj: blocco lungo x; bi: blocco lungo y; bk: blocco lungo z
    int bj = b % nbx, bi = (b / nbx) % nbx, bk = b / (nbx * nbx);
    // sinistra, destra, sotto, sopra e davanti, dietro
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};
    // verifico se si trova nella griglia
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // scarta blocchi fuori dalla griglia
        // scarta blocchi attivi e non convergenti
        if (in_list[nb[d]] && !Cb_step1[nb[d]]) continue;          // d: indica la direzione; N = 33; nb[d] = [61, 63, 57, 67, 37, 87];      
        if (atomicExch(&is_candidate[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(float* d_u, const float* d_F, const unsigned char* d_mask, int N, float h,
              const std::vector<int>& srcs, float eps, int niter, float* ms)
{
    // nbx: numero di blocchi per lato; n_blocks: blocchi totali
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx;
    // Cb: si inizializza vettore di n_blocks valori a 1
    std::vector<unsigned char> Cb(n_blocks, 1);
    // strato, riga e colonna del nodo sorgente
    for (int s : srcs){
    const int sk = s / (N * N), si = (s / N) % N, sj = s % N;
    // Cb: vale 1 tranne nel blocco della sorgente (0) 
    Cb[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;
    }
    // Memoria sulla GPU e copia dei dati iniziali (CPU -> GPU)
    unsigned char *d_Cb, *d_Cb_step1, *d_in_list; int *d_active_list, *d_n_active;
    int *d_is_candidate, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks)); CUDA_CHECK(cudaMalloc(&d_Cb_step1, n_blocks)); CUDA_CHECK(cudaMalloc(&d_in_list, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active_list, n_blocks * sizeof(int))); CUDA_CHECK(cudaMalloc(&d_n_active, sizeof(int)));
    // Memoria per i blocchi candidati
    CUDA_CHECK(cudaMalloc(&d_is_candidate, n_blocks * sizeof(int))); CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));
    // si copiano d_u, d_F e d_Cb dalla CPU alla GPU
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb.data(), n_blocks, cudaMemcpyHostToDevice));
    // si azzerano d_Cb_step1 e d_is_candidate
    CUDA_CHECK(cudaMemset(d_Cb_step1, 0, n_blocks)); CUDA_CHECK(cudaMemset(d_is_candidate, 0, n_blocks * sizeof(int)));
    // list_blocks: blocchi CUDA da 256 threads per avere un thread per ogni blocco di nodi 
    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;
    // si misura l'ora prima del ciclo
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start)); CUDA_CHECK(cudaEventCreate(&stop)); CUDA_CHECK(cudaEventRecord(start));                     
    int iter = 0;
    while (true) {
        // azzera il numero di blocchi attivi                                           
        int n_active = 0;  // contatore su CPU
        CUDA_CHECK(cudaMemset(d_n_active, 0, sizeof(int)));    // contatore su GPU
        // lancia list_blocks CUDA da 256 threads ciascuno, sufficiente per avere un thread ogni blocco
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_in_list, d_active_list, d_n_active, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        // aggiorna n_active con d_n_active
        CUDA_CHECK(cudaMemcpy(&n_active, d_n_active, sizeof(int), cudaMemcpyDeviceToHost));
        if (n_active == 0) break;  // se n_active == 0, lista A vuota, termina ciclo
        ++iter;                 // conta i giri
        // Step 1: foreach b in A in parallelo; lancia n_active blocchi 8x8x8
        update_blocks<<<n_active, dim3(BLOCK, BLOCK, BLOCK)>>>(d_active_list, nullptr, niter, d_u, d_F, N, nbx, h, d_mask, eps, d_Cb_step1, nullptr);
        CUDA_CHECK(cudaGetLastError());
        // azzeriamo il contatore dei candidati sulla GPU
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        // lancia n_active blocchi CUDA da 256 threads
        build_candidate_list<<<(n_active + threads-1) / threads, threads>>>(d_active_list, n_active, nbx, d_in_list, d_Cb_step1, d_Cb,
                                                                          d_is_candidate, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        // ogni blocco ha al più 6 vicini, quindi 6*n_active
        update_blocks<<<std::min(6 * n_active, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, 1,
                                                                                    d_u, d_F, N, nbx, h, d_mask, eps, d_Cb, d_is_candidate);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start)); CUDA_CHECK(cudaEventDestroy(stop));
    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaFree(d_Cb)); CUDA_CHECK(cudaFree(d_Cb_step1));
    CUDA_CHECK(cudaFree(d_in_list)); CUDA_CHECK(cudaFree(d_active_list)); CUDA_CHECK(cudaFree(d_n_active));
    CUDA_CHECK(cudaFree(d_is_candidate)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}

In [ ]:
%%writefile jacobi_fim.cuh
// FIM-Jacobi senza lista, in CUDA 3D: a ogni iterazione tutti i nodi si aggiornano con Godunov usando i valori
// dell'iterazione prima (Jacobi), nuovo = min(vecchio, Godunov); due buffer che si alternano
#pragma once
#include <vector>
#include <utility>          // std::swap
#include <cuda_runtime.h>
#include "godunov3d.h"  // risolutore locale

const int CONTROLLO = 25;   // ogni quante iterazioni si controlla la convergenza

// Un thread per nodo: nuovo valore del nodo x dai 6 vicini dell'iterazione precedente
__global__ void jacobi_update(const float* cur, const float* F, const unsigned char* mask, float* nxt,
                              int N, float h, float eps, int* cambiato)
{
    const int x = blockIdx.x * blockDim.x + threadIdx.x;
    if (x >= N * N * N) return;                                   // thread in eccesso
    const float p = cur[x];
    if (p == 0.0f || !mask[x]) { nxt[x] = p; return; }            // sorgente o ostacolo: non cambia (gli ostacoli restano a INF)
    const int j = x % N, i = (x / N) % N, k = x / (N * N);
    const float a = fminf(j > 0 ? cur[x - 1] : INF, j < N - 1 ? cur[x + 1] : INF);              // vicini lungo x
    const float b = fminf(i > 0 ? cur[x - N] : INF, i < N - 1 ? cur[x + N] : INF);              // vicini lungo y
    const float c = fminf(k > 0 ? cur[x - N * N] : INF, k < N - 1 ? cur[x + N * N] : INF);      // vicini lungo z
    const float q = fminf(p, solve_quadratic3(a, b, c, F[x], h));                               // min(vecchio, Godunov)
    nxt[x] = q;
    if (cambiato && p - q >= eps) *cambiato = 1;                  // solo nelle iterazioni di controllo: qualcuno cambia ancora
}

// Stessa interfaccia del FIM a nodi: le sorgenti sono gia' a 0 in u
int jacobi_fim(float* d_u, const float* d_F, const unsigned char* d_mask, int N, float h, float eps, float* ms)
{
    const int n = N * N * N;
    const size_t bytes = (size_t)n * sizeof(float);
    float* d_tmp; int* d_cambiato;                                  // d_tmp: il secondo buffer, interno
    CUDA_CHECK(cudaMalloc(&d_tmp, bytes)); CUDA_CHECK(cudaMalloc(&d_cambiato, sizeof(int)));
    float *d_cur = d_u, *d_nxt = d_tmp;                              // si parte dai valori in d_u
    const int threads = 256, blocchi = (n + threads - 1) / threads;   // come il suo threads_per_block = 256
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start)); CUDA_CHECK(cudaEventCreate(&stop)); CUDA_CHECK(cudaEventRecord(start));
    int it = 0;
    while (true) {
        const bool controlla = (it % CONTROLLO == 0);
        if (controlla) CUDA_CHECK(cudaMemset(d_cambiato, 0, sizeof(int)));
        jacobi_update<<<blocchi, threads>>>(d_cur, d_F, d_mask, d_nxt, N, h, eps, controlla ? d_cambiato : nullptr);
        CUDA_CHECK(cudaGetLastError());
        std::swap(d_cur, d_nxt);                                  // il nuovo diventa il vecchio
        ++it;
        if (controlla) {                                          // sincronizzazione GPU -> CPU solo ogni CONTROLLO iterazioni
            int cambiato = 0;
            CUDA_CHECK(cudaMemcpy(&cambiato, d_cambiato, sizeof(int), cudaMemcpyDeviceToHost));
            if (!cambiato) break;
        }
    }
    CUDA_CHECK(cudaEventRecord(stop)); CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));
    CUDA_CHECK(cudaEventDestroy(start)); CUDA_CHECK(cudaEventDestroy(stop));
    if (d_cur != d_u)                                                // il risultato e' rimasto nel buffer interno:
        CUDA_CHECK(cudaMemcpy(d_u, d_cur, bytes, cudaMemcpyDeviceToDevice));   // copia GPU -> GPU, molto veloce
    CUDA_CHECK(cudaFree(d_tmp)); CUDA_CHECK(cudaFree(d_cambiato));

    return it;
}

In [ ]:
%%writefile percorso.h
// Percorso del veicolo: ricostruzione con RK4 lungo -grad u, orario lungo il percorso e file per la grafica.
// Coordinate nel cubo [-1, 1]^3; il nodo (j, i, k) e' il punto (x, y, z) = (-1 + j h, -1 + i h, -1 + k h), indice q = (k * N + i) * N + j
#pragma once
#include <cstdio>          // printf, fopen
#include <cstdlib>         // exit
#include <vector>
#include <string>          // nome del file
#include <cmath>
#include <algorithm>       // std::clamp
#include "ambiente.h"      // Punto, A, B
#include "godunov3d.h"     // INF

// Operazioni sui punti (vettori 3D), per scrivere RK4 come sulla carta
inline Punto operator+(Punto a, Punto b) { return { a.x + b.x, a.y + b.y, a.z + b.z }; }
inline Punto operator-(Punto a, Punto b) { return { a.x - b.x, a.y - b.y, a.z - b.z }; }
inline Punto operator*(double s, Punto a) { return { s * a.x, s * a.y, s * a.z }; }
inline double norma(Punto a) { return std::sqrt(a.x * a.x + a.y * a.y + a.z * a.z); }

// Mappa: tutto quello che serve al veicolo per muoversi
struct Mappa {
    int N;                               // nodi per lato
    double h;                            // passo della griglia
    std::vector<float> u;                // tempo di arrivo da B (INF negli ostacoli)
    std::vector<float> F;                // lentezza
    std::vector<unsigned char> mask;     // 1 = libero, 0 = ostacolo
};
struct Cella { int j0, i0, k0; double wx, wy, wz; };   // cella che contiene un punto e posizione relativa (tra 0 e 1) dentro la cella
inline Cella cella(const Mappa& mp, Punto p)
{
    const double qx = (p.x + 1.0) / mp.h, qy = (p.y + 1.0) / mp.h, qz = (p.z + 1.0) / mp.h;   // posizione in indici, non intera
    Cella c;
    c.j0 = std::clamp((int)std::floor(qx), 0, mp.N - 2);
    c.i0 = std::clamp((int)std::floor(qy), 0, mp.N - 2);
    c.k0 = std::clamp((int)std::floor(qz), 0, mp.N - 2);
    c.wx = qx - c.j0; c.wy = qy - c.i0; c.wz = qz - c.k0;
    return c;
}

// Peso trilineare dell'angolo (j0 + dj, i0 + di, k0 + dk) della cella c: piu' p e' vicino a quell'angolo, piu' pesa
inline double peso(const Cella& c, int dj, int di, int dk)
{
    return (dj ? c.wx : 1 - c.wx) * (di ? c.wy : 1 - c.wy) * (dk ? c.wz : 1 - c.wz);
}

// Valore di f nel punto p, interpolato (trilineare) dagli 8 nodi della cella che contiene p
template <class T>
inline double interpola(const Mappa& mp, const std::vector<T>& f, Punto p)
{
    const Cella c = cella(mp, p);
    double s = 0.0;
    for (int dk = 0; dk <= 1; ++dk)
        for (int di = 0; di <= 1; ++di)
            for (int dj = 0; dj <= 1; ++dj)
                s += peso(c, dj, di, dk) * f[((c.k0 + dk) * mp.N + c.i0 + di) * mp.N + c.j0 + dj];
    return s;
}
// Vero se p non e' dentro un oggetto. Il percorso di tempo minimo rasenta il margine di sicurezza e, per l'errore di griglia,
// puo' entrarci di una frazione di h: si accetta, ma non deve mai entrare nell'oggetto vero
inline bool libero(const Mappa& mp, Punto p) { return !dentro_un_oggetto(p.x, p.y, p.z); }

// Vero se il nodo (j, i, k) e' nella griglia e u vi e' finita (falso nei nodi ostacolo, che dopo il calcolo restano a INF)
inline bool finito(const Mappa& mp, int j, int i, int k)
{
    if (j < 0 || j >= mp.N || i < 0 || i >= mp.N || k < 0 || k >= mp.N) return false;
    return mp.u[(k * mp.N + i) * mp.N + j] < INF;
}

// u nel punto p, interpolata dagli 8 nodi della cella ma solo con quelli finiti (vicino agli ostacoli i nodi a INF non contano):
// si sommano i nodi finiti e si divide per la somma dei loro pesi. INF se nessun nodo e' finito
inline double u_interpolata(const Mappa& mp, Punto p)
{
    const Cella c = cella(mp, p);
    double s = 0.0, peso_tot = 0.0;
    for (int dk = 0; dk <= 1; ++dk)
        for (int di = 0; di <= 1; ++di)
            for (int dj = 0; dj <= 1; ++dj) {
                const int j = c.j0 + dj, i = c.i0 + di, k = c.k0 + dk;
                if (!finito(mp, j, i, k)) continue;              // nodo ostacolo: non contribuisce
                const double w = peso(c, dj, di, dk);
                s += w * mp.u[(k * mp.N + i) * mp.N + j];
                peso_tot += w;
            }
    return peso_tot > 0.0 ? s / peso_tot : INF;
}
// Derivata di u in p lungo la direzione e (un passo h lungo un asse): differenza centrata se i punti p + e e p - e sono fuori
// dagli ostacoli; se uno dei due cade negli ostacoli (u = INF) si usa solo l'altro (differenza in avanti o all'indietro)
inline double derivata(const Mappa& mp, Punto p, Punto e)
{
    const double h = norma(e);
    const double u_avanti = u_interpolata(mp, p + e), u_indietro = u_interpolata(mp, p - e);
    const bool avanti = u_avanti < INF, indietro = u_indietro < INF;
    if (avanti && indietro) return (u_avanti - u_indietro) / (2.0 * h);   // differenza centrata
    const double u0 = u_interpolata(mp, p);
    if (avanti)   return (u_avanti - u0) / h;                             // solo in avanti
    if (indietro) return (u0 - u_indietro) / h;                           // solo all'indietro
    return 0.0;
}

// Gradiente di u in p: le tre derivate lungo x, y, z con un passo h
inline Punto gradiente(const Mappa& mp, Punto p)
{
    const double h = mp.h;
    return { derivata(mp, p, { h, 0.0, 0.0 }),
             derivata(mp, p, { 0.0, h, 0.0 }),
             derivata(mp, p, { 0.0, 0.0, h }) };
}

// Direzione di discesa in p: il versore -grad u / |grad u|, dove u (il tempo per arrivare a B) scende piu' in fretta.
// Falso se il gradiente e' nullo (non si sa dove andare)
inline bool direzione(const Mappa& mp, Punto p, Punto& d)
{
    const Punto g = gradiente(mp, p);
    const double ng = norma(g);
    if (ng < 1e-12) return false;
    d = (-1.0 / ng) * g;
    return true;
}

// PARTE 3: PERCORSO DEL VEICOLO, INTEGRANDO L'EQUAZIONE DEL RAGGIO CON RK4
// Percorso da p0 a B: si integra dp/ds = -grad u / |grad u| (l'equazione del raggio) con Runge-Kutta del 4 ordine a passo ds = h/2.
// Ci si ferma a meno di 2h da B e si chiude su B
inline std::vector<Punto> percorso(const Mappa& mp, Punto p0)
{
    const double ds = mp.h / 2.0;                            // passo di integrazione (lunghezza di ogni tratto)
    std::vector<Punto> P = { p0 };
    Punto p = p0;
    for (int passo = 0; passo < 20 * mp.N; ++passo) {        // limite di sicurezza sul numero di passi
        if (norma(p - B) < 2.0 * mp.h) {                     // vicino all'arrivo: si chiude su B
            P.push_back(B);
            return P;
        }
        Punto k1, k2, k3, k4;                                // le quattro direzioni di RK4
        if (!direzione(mp, p, k1) ||
            !direzione(mp, p + (ds / 2.0) * k1, k2) ||
            !direzione(mp, p + (ds / 2.0) * k2, k3) ||
            !direzione(mp, p + ds * k3, k4)) {
            printf("RK4: gradiente nullo in (%.3f, %.3f, %.3f), il percorso non prosegue\n", p.x, p.y, p.z);
            exit(1);
        }
        const Punto q = p + (ds / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4);   // passo RK4: media pesata delle quattro direzioni
        if (!libero(mp, q)) {
            printf("RK4: il passo da (%.3f, %.3f, %.3f) entrerebbe in un oggetto\n", p.x, p.y, p.z);
            exit(1);
        }
        P.push_back(q);
        p = q;
    }
    printf("RK4: B non raggiunto in %d passi (ultimo punto (%.3f, %.3f, %.3f))\n", 20 * mp.N, p.x, p.y, p.z);
    exit(1);
}

// PARTE 4: RELATIVA ALLE UNITA' FISICHE, ALLE VELOCITA' E ALL'ORARIO LUNGO IL PERCORSO
// UNITA' FISICHE: il cubo [-1, 1]^3 e' un tratto di mare di 20 m di lato; dove F = 1 il veicolo va a V_RIF
  const double METRI_PER_UNITA = 10.0;                     // 1 unita' di coordinata = 10 m
  const double V_RIF = 1.0;                                // velocita' [m/s] dove F = 1: in acqua da 0.2 m/s (fondo) a 1.8 m/s (superficie)
  const double SEC_PER_U = METRI_PER_UNITA / V_RIF;        // 1 unita' di u = 10 s

  // Orario lungo il percorso P: istante t[i] [s] in cui il veicolo passa dal punto P[i] e velocita' v[i] [m/s] in quel punto
  struct Orario { std::vector<double> t, v; };

  // Ogni tratto P[i-1] -> P[i] dura F (nel suo punto medio) x lunghezza x SEC_PER_U: sommati danno l'integrale di F ds, cioe' u(A),
  // in secondi. La velocita' in ogni punto e' V_RIF / F
  inline Orario orario(const Mappa& mp, const std::vector<Punto>& P)
  {
      Orario o{ { 0.0 }, {} };                                 // il veicolo parte all'istante 0
        for (size_t i = 1; i < P.size(); ++i) {
          const Punto medio = 0.5 * (P[i - 1] + P[i]);
          o.t.push_back(o.t.back() + interpola(mp, mp.F, medio) * norma(P[i] - P[i - 1]) * SEC_PER_U);
        }
        for (const Punto& p : P) o.v.push_back(V_RIF / interpola(mp, mp.F, p));
      return o;
  }

  // File dei percorsi per la cella Python della grafica: una riga per dato, con un'etichetta all'inizio
  inline void scrivi_percorsi(const std::string& nome, const std::vector<std::string>& metodi,
                              const std::vector<std::vector<Punto>>& percorsi, const std::vector<Orario>& orari)
  {
    FILE* f = fopen(nome.c_str(), "w");
    fprintf(f, "PARAMETRI %.17g %.17g %.17g %.17g %.17g %.17g\n", METRI_PER_UNITA, V_RIF, SEC_PER_U, V0, G, MARGINE);
    fprintf(f, "A %.17g %.17g %.17g\nB %.17g %.17g %.17g\n", A.x, A.y, A.z, B.x, B.y, B.z);
    for (const Sfera& s : OSTACOLI)                          // gli ostacoli sferici: centro e raggio
        fprintf(f, "SFERA %.17g %.17g %.17g %.17g\n", s.cx, s.cy, s.cz, s.r);
    for (size_t m = 0; m < metodi.size(); ++m) {             // un percorso per metodo: t x y z v per ogni punto
        fprintf(f, "METODO %s\n", metodi[m].c_str());
    for (size_t i = 0; i < percorsi[m].size(); ++i)
        fprintf(f, "PUNTO %.9g %.9g %.9g %.9g %.9g\n", orari[m].t[i],
        percorsi[m][i].x, percorsi[m][i].y, percorsi[m][i].z, orari[m].v[i]);
    }
    fclose(f);
}

In [ ]:
%%writefile main.cu
// Veicolo subacqueo in esplorazione: F assegnata (strati, termoclino, acqua densa) e ostacoli sferici; una sola iconale da B
// risolta con il FMM, il FIM-Jacobi e il Block FIM; confronto dei tre metodi, tempi e percorso del veicolo con RK4
#include <cstdio>
#include <chrono>          // tempi sulla CPU
#include <string>          // nome del file dei percorsi
#include "fmm3d.h"         // Fast Marching (CPU)
#include "ambiente.h"      // acqua, oggetti, maschera, lentezza
#include "block_fim3d.cuh" // Block FIM (GPU)
#include "jacobi_fim.cuh"  // FIM-Jacobi (GPU)
#include "percorso.h"      // percorso del veicolo (RK4), orario, file dei percorsi per la grafica

enum Metodo { FMM, FIM_JACOBI, BLOCK_FIM };                  // con quale metodo si risolve l'iconale da B
const char* NOMI[3] = { "FMM", "FIM-Jacobi", "Block FIM" };

// Tempo di arrivo u da B: una sola iconale |grad u| = F, con F assegnata. Maschera e F sono preparate sulla CPU;
// per i metodi su GPU si copiano sulla GPU insieme a u iniziale (INF ovunque, 0 in B) e u torna sulla CPU alla fine.
// t_ms = solo i cicli di calcolo; ttot_ms = tutto (allocazione, copie e calcolo)
int tempo_di_arrivo(Metodo metodo, int N, const std::vector<unsigned char>& mask, const std::vector<float>& F,
                    float eps, int niter, std::vector<float>& u, double& t_ms, double& ttot_ms)
{
    const auto t_inizio = std::chrono::high_resolution_clock::now();
    const float h = 2.0f / (N - 1);
    const int n = N * N * N;
    const std::vector<int> srcs = { nodo(N, B.x, B.y, B.z) };       // la sorgente e' B
    u.assign(n, INF);
    for (int s : srcs) u[s] = 0.0f;
    int giri = 0;
    if (metodo == FMM) {                                     // CPU
        const auto t0 = std::chrono::high_resolution_clock::now();
        fmm(u, F, N, h, srcs, mask);
        const auto t1 = std::chrono::high_resolution_clock::now();
        t_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
    } else {                                                 // GPU (FIM-Jacobi o Block FIM)
        const size_t bytes = (size_t)n * sizeof(float);
        float *d_u, *d_F; unsigned char* d_mask;
        CUDA_CHECK(cudaMalloc(&d_u, bytes)); CUDA_CHECK(cudaMalloc(&d_F, bytes)); CUDA_CHECK(cudaMalloc(&d_mask, n));
        CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), n, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
        float ms = 0.0f;
        if (metodo == BLOCK_FIM) giri = block_fim(d_u, d_F, d_mask, N, h, srcs, eps, niter, &ms);
        else                     giri = jacobi_fim(d_u, d_F, d_mask, N, h, eps, &ms);
        CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F)); CUDA_CHECK(cudaFree(d_mask));
        t_ms = ms;                                           // solo i cicli di calcolo
    }
    const auto t_fine = std::chrono::high_resolution_clock::now();
    ttot_ms = std::chrono::duration<double, std::milli>(t_fine - t_inizio).count();
    return giri;
}

// File per la grafica: tempi di calcolo dei tre metodi (nella gara sono il ritardo con cui parte ogni veicolo) e la lentezza F
// sulla sezione verticale che contiene A e B (n x n punti): x, y, z, F; F = nan dentro le sfere (li' si vede l'ostacolo grigio)
void scrivi_scenario(const std::string& nome, int n, const double t_calcolo[3])
{
    FILE* f = fopen(nome.c_str(), "w");
    for (int m = 0; m < 3; ++m) fprintf(f, "CALCOLO %s %.6g\n", NOMI[m], t_calcolo[m]);     // [ms]
    // Piano verticale per A e B: punti (x, y) = A + s d, con d la direzione orizzontale da A a B; s va da bordo a bordo del cubo
    const double lx = B.x - A.x, ly = B.y - A.y, l = std::sqrt(lx * lx + ly * ly);
    const double dx = lx / l, dy = ly / l;
    double s0 = -1e9, s1 = 1e9;
    if (std::fabs(dx) > 1e-12) { const double a = (-1.0 - A.x) / dx, b = (1.0 - A.x) / dx; s0 = std::max(s0, std::min(a, b)); s1 = std::min(s1, std::max(a, b)); }
    if (std::fabs(dy) > 1e-12) { const double a = (-1.0 - A.y) / dy, b = (1.0 - A.y) / dy; s0 = std::max(s0, std::min(a, b)); s1 = std::min(s1, std::max(a, b)); }
    fprintf(f, "SEZIONE %d\n", n);
    for (int k = 0; k < n; ++k)                              // righe: quota z; colonne: posizione s lungo il piano
        for (int c = 0; c < n; ++c) {
            const double s = s0 + (s1 - s0) * c / (n - 1), z = -1.0 + 2.0 * k / (n - 1);
            const double x = A.x + s * dx, y = A.y + s * dy;
            if (dentro_un_oggetto(x, y, z)) fprintf(f, "%.6g %.6g %.6g nan\n", x, y, z);
            else                            fprintf(f, "%.6g %.6g %.6g %.6g\n", x, y, z, lentezza(x, y, z));
        }
    fclose(f);
}

int main()
{
    // Informazioni sulla GPU usata
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d\n", prop.name, prop.multiProcessorCount);
    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    const int niter = 8;                                     // iterazioni locali massime del Block FIM (uscita anticipata)
    if (!ambiente_valido()) {
        printf("Errore: A o B sono dentro un oggetto, correggi ambiente.h\n");
        return 1;
    }
    printf("Velocita' dell'acqua: %.2f in A, %.2f in B, %.2f al centro del termoclino sopra A (m/s, con V_RIF = %.1f)\n",
           velocita(A.x, A.y, A.z) * V_RIF, velocita(B.x, B.y, B.z) * V_RIF, velocita(A.x, A.y, Z_TERMO) * V_RIF, V_RIF);

    // Al variare di N: i risultati si salvano nel ciclo e si stampano dopo, in quattro tabelle con le stesse righe
    const std::vector<int> GRIGLIE = { 128, 256, 384 };
    struct Riga { int N; double diff[2], uA[3], lung, t_perc; size_t punti; double t[3], ttot[3]; };
    std::vector<Riga> righe;
    for (int N : GRIGLIE) {
        const std::vector<unsigned char> mask = costruisci_maschera(N);   // 1 = libero, 0 = oggetto (con il MARGINE)
        const std::vector<float> F = costruisci_lentezza(N);              // F = 1/v
        const double h = 2.0 / (N - 1);
        Riga r{}; r.N = N;
        std::vector<Mappa> mappe;                                          // una mappa risolta per metodo
        for (int m = 0; m < 3; ++m) {
            std::vector<float> u;
            tempo_di_arrivo((Metodo)m, N, mask, F, eps, niter, u, r.t[m], r.ttot[m]);
            mappe.push_back({ N, h, std::move(u), F, mask });
        }
        // Accuratezza: differenza massima dalla u del Block FIM su tutti i nodi liberi; u in A interpolata (A non e' un nodo)
        for (int m = 0; m < 2; ++m) {
            double d = 0.0;
            for (size_t q = 0; q < mask.size(); ++q)
                if (mask[q]) d = std::max(d, std::fabs((double)mappe[m].u[q] - mappe[BLOCK_FIM].u[q]));
            r.diff[m] = d;
        }
        for (int m = 0; m < 3; ++m) r.uA[m] = u_interpolata(mappe[m], A);
        // Percorso del veicolo sulla u del Block FIM: numero di punti, lunghezza e tempo di percorrenza
        const std::vector<Punto> P = percorso(mappe[BLOCK_FIM], A);
        const Orario o = orario(mappe[BLOCK_FIM], P);
        r.punti = P.size(); r.t_perc = o.t.back(); r.lung = 0.0;
        for (size_t i = 1; i < P.size(); ++i) r.lung += norma(P[i] - P[i - 1]) * METRI_PER_UNITA;
        righe.push_back(r);
        // Sulla griglia piu' fine: i percorsi dei tre metodi (ognuno sulla propria u), la lentezza e i tempi per la grafica
        if (N == GRIGLIE.back()) {
            std::vector<std::vector<Punto>> percorsi;
            std::vector<Orario> orari;
            for (int m = 0; m < 3; ++m) {
                percorsi.push_back(percorso(mappe[m], A));
                orari.push_back(orario(mappe[m], percorsi.back()));
            }
            scrivi_percorsi("percorsi_" + std::to_string(N) + ".txt", { NOMI[0], NOMI[1], NOMI[2] }, percorsi, orari);
            scrivi_scenario("scenario_" + std::to_string(N) + ".txt", 100, r.t);
        }
    }

    // 1) Accuratezza: i tre metodi danno la stessa u?
    printf("\nAccuratezza: max|...| = differenza massima dalla u del Block FIM su tutti i nodi liberi; u(A) interpolata [unita' di u]\n");
    printf("    N   max|FMM-BFIM|   max|J-FIM-BFIM|   u(A) FMM   u(A) J-FIM   u(A) BFIM\n");
    for (const Riga& r : righe)
        printf("%5d   %13.2e   %15.2e   %8.5f   %10.5f   %9.5f\n", r.N, r.diff[0], r.diff[1], r.uA[0], r.uA[1], r.uA[2]);
    // 2) Percorso del veicolo (sulla u del Block FIM)
    printf("\nPercorso (RK4 sulla u del Block FIM); la retta da A a B e' lunga %.2f m\n", norma(B - A) * METRI_PER_UNITA);
    printf("    N   punti   lunghezza [m]   t percorso [s]   u(A) [s]\n");
    for (const Riga& r : righe)
        printf("%5d   %5zu   %13.2f   %14.2f   %8.2f\n", r.N, r.punti, r.lung, r.t_perc, r.uA[2] * SEC_PER_U);
    // 3) Tempo di calcolo: solo i cicli dell'iconale (e' il ritardo di partenza nella gara)
    printf("\nTempo di calcolo [ms]: solo i cicli dell'iconale da B; speedup = t metodo / t Block FIM\n");
    printf("    N   t FMM [ms]   t J-FIM [ms]   t BFIM [ms]   sp vs FMM   sp vs J-FIM\n");
    for (const Riga& r : righe)
        printf("%5d   %10.1f   %12.2f   %11.2f   %9.1f   %11.1f\n", r.N, r.t[0], r.t[1], r.t[2], r.t[0] / r.t[2], r.t[1] / r.t[2]);
    // 4) Tempo totale: anche allocazione e copie CPU <-> GPU
    printf("\nTempo totale [ms]: allocazione, copie CPU <-> GPU e calcolo\n");
    printf("    N   t FMM [ms]   t J-FIM [ms]   t BFIM [ms]   sp vs FMM   sp vs J-FIM\n");
    for (const Riga& r : righe)
        printf("%5d   %10.1f   %12.2f   %11.2f   %9.1f   %11.1f\n", r.N, r.ttot[0], r.ttot[1], r.ttot[2],
               r.ttot[0] / r.ttot[2], r.ttot[1] / r.ttot[2]);
    // 5) Gara: ogni veicolo parte da A quando il suo metodo ha finito i cicli di calcolo (N piu' fine)
    const Riga& u = righe.back();
    printf("\nGara (N = %d): ogni veicolo parte quando il suo metodo ha finito il calcolo\n", u.N);
    for (int m = 0; m < 3; ++m)
        printf("  %-10s  parte dopo %7.2f s, arriva in B dopo %7.2f s\n", NOMI[m], u.t[m] / 1000.0, u.t[m] / 1000.0 + u.t_perc);
    return 0;
}

In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_75 main.cu -o fim3d && ./fim3d

In [ ]:
# VEICOLO SUBACQUEO - GRAFICA: sezione verticale della lentezza F, ostacoli sferici, percorso colorato con la velocita'
# e gara dei tre veicoli (uno per metodo: ognuno parte da A quando il suo metodo ha finito il calcolo); sotto, la velocita' v(t).
# Legge i file scritti da main.cu (percorsi_N.txt e scenario_N.txt). Si cambia solo per l'aspetto: colori, dimensioni.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

N = 384                                                             # griglia piu' fine di main.cu (l'ultima di GRIGLIE)
METODI = ["FMM", "FIM-Jacobi", "Block FIM"]
colore = {"FMM": "#52514e", "FIM-Jacobi": "#eb6834", "Block FIM": "#2a78d6"}   # come negli altri notebook
N_FRAME = 100                                                       # fotogrammi dell'animazione (al massimo 120)
DURATA_ANIMAZIONE = 10.0                                            # durata del Play [s], qualunque sia la durata vera

# 1) LETTURA DEI FILE
sfere, percorsi = [], {}
for riga in open(f"percorsi_{N}.txt"):
    k, *v = riga.split()
    if k == "METODO": met = " ".join(v); percorsi[met] = []; continue
    v = [float(t) for t in v]
    if k == "PARAMETRI": METRI_PER_UNITA, V_RIF, SEC_PER_U, V0, G, MARGINE = v
    elif k == "A": A = np.array(v)
    elif k == "B": B = np.array(v)
    elif k == "SFERA": sfere.append(v)
    elif k == "PUNTO": percorsi[met].append(v)
percorsi = {met: np.array(p) for met, p in percorsi.items()}         # colonne: t, x, y, z, v (t da quando il veicolo parte)
ritardo, sezione = {}, []
for riga in open(f"scenario_{N}.txt"):
    k, *v = riga.split()
    if k == "CALCOLO": ritardo[" ".join(v[:-1])] = float(v[-1]) / 1000.0   # tempo di calcolo [s] = ritardo di partenza
    elif k == "SEZIONE": n = int(v[0])
    else: sezione.append([float(t) for t in [k] + v])
sezione = np.array(sezione).reshape(n, n, 4)                        # [riga = quota z, colonna = posizione nel piano, (x, y, z, F)]
for met in METODI:
    print(f"{met:10s}: parte dopo {ritardo[met]:7.2f} s di calcolo, viaggia {percorsi[met][-1, 0]:6.2f} s, "
          f"arriva in B dopo {ritardo[met] + percorsi[met][-1, 0]:7.2f} s")

# 2) PARTE FISSA (disegnata una volta sola): sezione di F, sfere, percorso colorato con la velocita'; coordinate in metri
m = METRI_PER_UNITA
fig = make_subplots(rows=2, cols=1, specs=[[{"type": "scene"}], [{"type": "xy"}]], row_heights=[0.72, 0.28], vertical_spacing=0.04)
F = sezione[:, :, 3]
fig.add_trace(go.Surface(x=m * sezione[:, :, 0], y=m * sezione[:, :, 1], z=m * sezione[:, :, 2], surfacecolor=F,
                         colorscale="Blues", cmin=np.nanmin(F), cmax=np.nanmax(F), opacity=0.85, showlegend=False, hoverinfo="skip",
                         colorbar=dict(title=dict(text="Lentezza F (alta = lento)", side="right"), x=0.98, y=0.75, len=0.5)), row=1, col=1)
th, ph = np.meshgrid(np.linspace(0, np.pi, 30), np.linspace(0, 2 * np.pi, 40))
for cx, cy, cz, r in sfere:                                         # ogni sfera: superficie grigia semitrasparente
    fig.add_trace(go.Surface(x=m * (cx + r * np.sin(th) * np.cos(ph)), y=m * (cy + r * np.sin(th) * np.sin(ph)),
                             z=m * (cz + r * np.cos(th)), colorscale=[[0, "grey"], [1, "grey"]], showscale=False,
                             opacity=0.35, showlegend=False, hoverinfo="skip"), row=1, col=1)
p = percorsi["Block FIM"]                                           # i tre percorsi coincidono: si disegna quello del Block FIM
fig.add_trace(go.Scatter3d(x=m * p[:, 1], y=m * p[:, 2], z=m * p[:, 3], mode="lines", showlegend=False, hoverinfo="skip",
                           line=dict(color=p[:, 4], colorscale="Plasma", width=8,
                                     colorbar=dict(title=dict(text="Velocità [m/s]", side="right"), x=1.08, y=0.75, len=0.5))), row=1, col=1)

# 3) VELOCITA' NEL TEMPO: ogni veicolo e' fermo (v = 0) mentre il suo metodo calcola e dopo l'arrivo in B
fine = max(ritardo[met] + percorsi[met][-1, 0] for met in METODI)
def curva_v(met):
    p = percorsi[met]
    arrivo = ritardo[met] + p[-1, 0]
    t = np.concatenate([[0.0, ritardo[met]], ritardo[met] + p[:, 0], [arrivo, fine]])
    v = np.concatenate([[0.0, 0.0], p[:, 4], [0.0, 0.0]])
    return t, v
for met in METODI:
    t, v = curva_v(met)
    fig.add_trace(go.Scatter(x=t, y=v, mode="lines", line=dict(color=colore[met], width=1.5), showlegend=False, hoverinfo="skip"), row=2, col=1)

# 4) PARTE MOBILE: il veicolo di ogni metodo (fermo in A mentre calcola, poi lungo il percorso, poi fermo in B) e il suo punto su v(t)
def stato(met, t):
    p = percorsi[met]
    tv = t - ritardo[met]                                           # tempo di viaggio trascorso
    if tv < 0.0 or tv > p[-1, 0]: v = 0.0                           # fermo: calcola, oppure e' arrivato
    else: v = np.interp(tv, p[:, 0], p[:, 4])
    tv = np.clip(tv, 0.0, p[-1, 0])
    return [np.interp(tv, p[:, 0], p[:, j]) for j in (1, 2, 3)], v
def mobili(t):
    tracce = []
    for met in METODI:
        (x, y, z), _ = stato(met, t)
        tracce.append(go.Scatter3d(x=[m * x], y=[m * y], z=[m * z], mode="markers", name=f"veicolo {met}",
                                   marker=dict(size=6, color=colore[met])))
    for met in METODI:
        _, v = stato(met, t)
        tracce.append(go.Scatter(x=[t], y=[v], mode="markers", showlegend=False, marker=dict(size=8, color=colore[met])))
    return tracce
iniziali = mobili(0.0)
for tr in iniziali[:3]: fig.add_trace(tr, row=1, col=1)
for tr in iniziali[3:]: fig.add_trace(tr, row=2, col=1)
indici = list(range(len(fig.data) - 6, len(fig.data)))             # le sole tracce che cambiano nei fotogrammi

# 5) ANIMAZIONE: N_FRAME istanti uniformi nel tempo t, Play di DURATA_ANIMAZIONE secondi in tutto
istanti = np.linspace(0.0, fine, N_FRAME)
ms_frame = 1000.0 * DURATA_ANIMAZIONE / N_FRAME
fig.frames = [go.Frame(data=mobili(t), traces=indici, name=f"{i}") for i, t in enumerate(istanti)]
fig.update_layout(height=1000, margin=dict(l=0, r=100, t=20, b=90), legend=dict(x=0.01, y=0.99),
                  scene=dict(xaxis=dict(title="x [m]", range=[-m, m]), yaxis=dict(title="y [m]", range=[-m, m]),
                             zaxis=dict(title="quota z [m]", range=[-m, m]), aspectmode="cube"),
                  xaxis=dict(title="t [s]", range=[0, fine]), yaxis=dict(title="v [m/s]"),
                  updatemenus=[dict(type="buttons", x=0.0, y=-0.09, xanchor="left", yanchor="top", direction="right", buttons=[
                      dict(label="Play", method="animate",
                           args=[None, dict(frame=dict(duration=ms_frame, redraw=True), transition=dict(duration=0), fromcurrent=True)]),
                      dict(label="Pausa", method="animate",
                           args=[[None], dict(frame=dict(duration=0, redraw=False), transition=dict(duration=0), mode="immediate")])])],
                  sliders=[dict(x=0.15, len=0.85, y=-0.07, yanchor="top", currentvalue=dict(prefix="", xanchor="left"),
                                steps=[dict(method="animate", label=f"t = {t:.1f} s",
                                            args=[[f"{i}"], dict(mode="immediate", frame=dict(duration=0, redraw=True),
                                                                 transition=dict(duration=0))])
                                       for i, t in enumerate(istanti)])])
fig.write_html("subacqueo_gara.html", include_plotlyjs="cdn")
fig.show()

In [ ]:
# PROFILO con Nsight Systems : dove va il tempo sulla GPU (kernel, copie CPU <-> GPU, chiamate CUDA).
# Rilancia tutto il programma (FMM compreso). I tempi stampati in questo giro sono un po' rallentati dal profiler.
!nsys profile --stats=true --force-overwrite true -o profilo_subacqueo ./fim3d